In [2]:
!pip install -q sentence-transformers faiss-cpu

from sentence_transformers import SentenceTransformer
import faiss


texts = [
    "نحتاج مراجعة أمنية قبل إطلاق النظام",
    "تعرضنا لمحاولة اختراق على البريد الإلكتروني",
    "أحتاج موعد مع المهندس سلطان لمراجعة الخطة",
    "عرض السعر المرسل أعلى من الميزانية",
    "وش صار على مشروع الأتمتة؟"
]
model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

embeddings = model.encode(texts)

index = faiss.IndexFlatL2(embeddings.shape[1])

index.add(embeddings)

print("عدد النصوص في FAISS:", index.ntotal)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

عدد النصوص في FAISS: 5


In [3]:
query = "فيه أحد حاول يدخل على إيميلاتنا"

query_embedding = model.encode([query])

distances, indices = index.search(query_embedding, 2)

print("Query:", query)
print("Results:")

for i in indices[0]:
    print("-", texts[i])

Query: فيه أحد حاول يدخل على إيميلاتنا
Results:
- تعرضنا لمحاولة اختراق على البريد الإلكتروني
- وش صار على مشروع الأتمتة؟


In [4]:
results = [
    "نحتاج مراجعة أمنية قبل إطلاق النظام",
    "تعرضنا لمحاولة اختراق على البريد الإلكتروني",
    "وش صار على مشروع الأتمتة؟"
]

correct_answer = "تعرضنا لمحاولة اختراق على البريد الإلكتروني"

rank = results.index(correct_answer) + 1

recall_at_10 = 1 if rank <= 10 else 0
mrr_at_10 = 1 / rank if rank <= 10 else 0

print("Rank:", rank)
print("Recall@10:", recall_at_10)
print("MRR@10:", mrr_at_10)

Rank: 2
Recall@10: 1
MRR@10: 0.5


In [5]:
from sentence_transformers import CrossEncoder

query = "فيه أحد حاول يدخل على إيميلاتنا"

results = [
    "نحتاج مراجعة أمنية قبل إطلاق النظام",
    "تعرضنا لمحاولة اختراق على البريد الإلكتروني",
    "وش صار على مشروع الأتمتة؟"
]

reranker = CrossEncoder(
    "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
)

pairs = [[query, text] for text in results]

scores = reranker.predict(pairs)

print("Query:", query)
print("\nBefore reranking:")
for text, score in zip(results, scores):
    print(round(float(score), 2), "→", text)

print("\nAfter reranking:")
for text, score in sorted(zip(results, scores), key=lambda x: x[1], reverse=True):
    print(round(float(score), 2), "→", text)

config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Query: فيه أحد حاول يدخل على إيميلاتنا

Before reranking:
-6.81 → نحتاج مراجعة أمنية قبل إطلاق النظام
-4.52 → تعرضنا لمحاولة اختراق على البريد الإلكتروني
-6.93 → وش صار على مشروع الأتمتة؟

After reranking:
-4.52 → تعرضنا لمحاولة اختراق على البريد الإلكتروني
-6.81 → نحتاج مراجعة أمنية قبل إطلاق النظام
-6.93 → وش صار على مشروع الأتمتة؟


**Lab6**

In [1]:
project_name = "DanaShift"

requests = [
    {"text": "وش صار على مشروع الأتمتة؟",           "language": "Arabic",  "dialect": "Saudi",    "length": "Short", "correct": 1},
    {"text": "أبغى أعرف وين وصل طلب الاجتماع مع المهندس سلطان لأن الموعد قرب", "language": "Arabic",  "dialect": "Saudi",    "length": "Long",  "correct": 1},
    {"text": "عايز أعرف المشروع وصل لفين",           "language": "Arabic",  "dialect": "Egyptian", "length": "Short", "correct": 0},
    {"text": "ما هي آخر مستجدات المراجعة الأمنية؟",   "language": "Arabic",  "dialect": "MSA",      "length": "Short", "correct": 1},
    {"text": "Please send the final proposal",       "language": "English", "dialect": "Standard", "length": "Short", "correct": 1},
    {"text": "We need a full security review of all our systems before the launch next month", "language": "English", "dialect": "Standard", "length": "Long",  "correct": 0}
]

print("🧪 Project:", project_name, "\n")

for r in requests:
    print(
        r["language"],
        "|", r["dialect"],
        "|", r["length"],
        "| Correct:", r["correct"],
        "|", r["text"]
    )

# 📊 الدقة لكل فئة
print("\n📊 Accuracy by group:")
for key in ["language", "dialect", "length"]:
    groups = {}
    for r in requests:
        groups.setdefault(r[key], []).append(r["correct"])
    for name, values in groups.items():
        acc = sum(values) / len(values)
        print(f"  {key:<9} {name:<9} → {acc:.0%}  ({sum(values)}/{len(values)})")

🧪 Project: DanaShift 

Arabic | Saudi | Short | Correct: 1 | وش صار على مشروع الأتمتة؟
Arabic | Saudi | Long | Correct: 1 | أبغى أعرف وين وصل طلب الاجتماع مع المهندس سلطان لأن الموعد قرب
Arabic | Egyptian | Short | Correct: 0 | عايز أعرف المشروع وصل لفين
Arabic | MSA | Short | Correct: 1 | ما هي آخر مستجدات المراجعة الأمنية؟
English | Standard | Short | Correct: 1 | Please send the final proposal
English | Standard | Long | Correct: 0 | We need a full security review of all our systems before the launch next month

📊 Accuracy by group:
  language  Arabic    → 75%  (3/4)
  language  English   → 50%  (1/2)
  dialect   Saudi     → 100%  (2/2)
  dialect   Egyptian  → 0%  (0/1)
  dialect   MSA       → 100%  (1/1)
  dialect   Standard  → 50%  (1/2)
  length    Short     → 75%  (3/4)
  length    Long      → 50%  (1/2)


In [2]:
import numpy as np

from sklearn.metrics import accuracy_score


y_true = np.array([1, 0, 1, 1, 0, 1, 0, 1])

y_pred = np.array([1, 0, 1, 0, 0, 1, 1, 1])


scores = []


for _ in range(1000):

    i = np.random.choice(len(y_true), len(y_true), replace=True)

    scores.append(accuracy_score(y_true[i], y_pred[i]))


print("95% CI:", np.percentile(scores, 2.5), "-", np.percentile(scores, 97.5))

95% CI: 0.375 - 1.0


In [6]:
errors = [
    ("فيه أحد حاول يدخل على إيميلاتنا", "أمن سيبراني", "أمن سيبراني"),
    ("كم بيكلفنا مشروع الأتمتة؟", "عروض أسعار", "أتمتة"),
    ("أبي أقابل المهندس سلطان بكرة", "اجتماعات", "اجتماعات"),
    ("السيرفر طاح ونبي ننقله للسحابة", "سحابة", "أمن سيبراني"),
    ("وش أفضل مطعم في الرياض؟", "خارج النطاق", "اجتماعات")
]

for text, actual, predicted in errors:
    print("الطلب:", text)
    print("التصنيف الصحيح:", actual)
    print("توقع النموذج:", predicted)
    print("التقييم:", "صحيح ✅" if actual == predicted else "خطأ ❌")
    print("-" * 30)

الطلب: فيه أحد حاول يدخل على إيميلاتنا
التصنيف الصحيح: أمن سيبراني
توقع النموذج: أمن سيبراني
التقييم: صحيح ✅
------------------------------
الطلب: كم بيكلفنا مشروع الأتمتة؟
التصنيف الصحيح: عروض أسعار
توقع النموذج: أتمتة
التقييم: خطأ ❌
------------------------------
الطلب: أبي أقابل المهندس سلطان بكرة
التصنيف الصحيح: اجتماعات
توقع النموذج: اجتماعات
التقييم: صحيح ✅
------------------------------
الطلب: السيرفر طاح ونبي ننقله للسحابة
التصنيف الصحيح: سحابة
توقع النموذج: أمن سيبراني
التقييم: خطأ ❌
------------------------------
الطلب: وش أفضل مطعم في الرياض؟
التصنيف الصحيح: خارج النطاق
توقع النموذج: اجتماعات
التقييم: خطأ ❌
------------------------------


In [5]:
report = {

    "Sliced Evaluation": "Language, Dialect, Length",

    "Confidence Interval": "95%",

    "Error Analysis": "Completed",

    "Error Taxonomy": "Completed"

}


for key, value in report.items():

    print(key, ":", value)

Sliced Evaluation : Language, Dialect, Length
Confidence Interval : 95%
Error Analysis : Completed
Error Taxonomy : Completed
